In [13]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [14]:
organism = "homo_sapiens"
cell_source = "placenta"
cell_state = None
table_name = "clean_degs.xlsx" # local name

## Get sheet names

In [15]:
metadata = pd.read_json("../metadata.json")
sheet_names = [obj["sheet_name"] for obj in metadata["tables"]]

sheet_names

['Marker_Genes_Table']

## Read in file

In [5]:
excel = pd.read_excel(table_name, sheet_name = sheet_names, skiprows = 1)

for e in list(excel.values()):
    print(e.columns)

Index(['gene', 'myAUC', 'avg_diff', 'power', 'cell type'], dtype='object')


In [6]:
cols = [
    {
        "sheet_name": sheet_name,
        "cols": {
            "avg_diff": "logfc",
            "gene": "feature_name",
            "cell type": "cell_type_label"
        }
    } for sheet_name in sheet_names]

In [7]:
for idx, sheet in enumerate(cols):
    sname = sheet["sheet_name"]
    print(sname)
    excel_name = excel
    if idx < 2:
        excel_name = excel
    # only keep the columns we want
    excel_name[sname] = excel_name[sname].loc[:,sheet["cols"].keys()].rename(columns=sheet["cols"])
    # add sheet name to the dataframe
    excel_name[sname]["sheet_name"] = sname

Marker_Genes_Table


In [10]:
df = pd.concat(list(excel.values()))
df["p_val"] = None
df["p_corr"] = None

df.head()

,logfc,feature_name,cell_type_label,sheet_name,p_val,p_corr
0,3.045510,INSL4,CTB_8W_1,Marker_Genes_Table,None,None
1,3.238270,MUC15,CTB_8W_1,Marker_Genes_Table,None,None
2,2.839081,TBX3,CTB_8W_1,Marker_Genes_Table,None,None
3,2.901008,KRT23,CTB_8W_1,Marker_Genes_Table,None,None
4,3.231121,SLC40A1,CTB_8W_1,Marker_Genes_Table,None,None


## Unfiltered

In [11]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None 

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": row["sheet_name"],
            "source_metrics": {
                "p_corr": row["p_corr"],
                "log_fc": row["logfc"],
            }
            }
        })

result[:1]


[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'INSL4',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'INSL4',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'Marker_Genes_Table',
   'source_metrics': {'p_corr': None, 'log_fc': 3.04550976409158}}}]

In [12]:
with open("evidence_unfiltered_metrics.json", "w") as f:
    json.dump(result, f, indent = 4) 

## Perform the filtering

In [17]:
col_pval = None
col_lfc = "avg_diff"
col_gene = "gene"
col_ct = "celltype"
col_rank = None
col_auc = "myAUC"
col_power = "power"

In [18]:
min_mean = 100
max_pval = 0.8
min_lfc = 3.52
max_gene_shares = 2
max_per_celltype = 20
min_auc = 0.7
min_power = 0.8

# filter by power and AUC 
dfc = df.query(f"{col_auc} >= {min_auc} & {col_power} >= {min_power} & {col_lfc} >= {min_lfc}")


# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]# .sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [19]:
markers[col_ct].value_counts()

celltype
STB_8W      20
Macro_1     20
Mes_2       20
Blood       20
Macro_2     12
Mes_1       10
CTB_8W_1     1
Name: count, dtype: int64

In [20]:
markers.groupby(col_ct)[col_gene].apply(lambda x: list(x))

celltype
Blood       [TMEM56, PKLR, MT1G4, NFE2, SPTA1, KEL, GFI1B,...
CTB_8W_1                                             [SLC1A2]
Macro_1     [RGS1, ITGAX, SERPINA1, CECR1, MSR1, LILRB4, H...
Macro_2     [VSIG4, MAF, LYVE1, F13A1, FOLR2, MRC1, CD14, ...
Mes_1       [IGFBP7, C1R, RARRES2, SPARCL1, IGFBP5, CFD3, ...
Mes_2       [WNT2, PAMR1, HSD17B2, SPON1, CXCL14, RSPO3, P...
STB_8W      [LGALS13, GH2, HOPX, PSG10P, CGB3, LINC00967, ...
Name: gene, dtype: object

In [21]:
markers.groupby(col_ct)[col_lfc].mean().sort_values()

celltype
Macro_1     3.987992
CTB_8W_1    4.001171
Blood       4.180594
Macro_2     4.231178
Mes_2       4.487878
STB_8W      4.499354
Mes_1       4.633982
Name: avg_diff, dtype: float64

## Convert to evidence json



In [26]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'SLC1A2',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'CTB_8W_1',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'SLC1A2',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000110436'},
  'source': {'source_type': 'deg',
   'source_rationale': 'filtered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'STB_8W',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'LGALS13',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'STB_8W',
   'cell_source': 'placenta',
   'cell_state': None,
   'feature_name': 'LGALS13',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000105198'},
  'source': {'s

## Save evidence

In [27]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 